In [ ]:
from Import_loader import *

In [ ]:
# Measurement
msr = measurement.Real("calibrated64/sphere_40mm_pos_4_foam_64f_330_mm")

In [ ]:
# Reconstruction Settings
config['recon_grid_samples'] = [4000, 10, 2000]
#config['recon_grid_samples'] = [200, 200, 100]
config['recon_grid_lims'] = [-0.187, 0.187, 0.0, 0.083, 0.25, 0.35]
config['recon_cal'] = True
config['recon_short'] = 0.345
config['recon_grid_normalize'] = True

config['recon_min_db'] = -15
config['recon_max_db'] = 0

In [ ]:
# Reconstruction
grid = msr.reconstruct()

In [ ]:
import torch
import torch.nn.functional as F

config['object'] = 'plane'

def find_local_maxima(tensor):
    padded = F.pad(tensor, (1, 1, 1, 1, 1, 1), mode='constant', value=float('-inf'))
    max_pooled = F.max_pool3d(padded.unsqueeze(0).unsqueeze(0), kernel_size=3, stride=1, padding=0).squeeze()
    local_max_mask = (tensor == max_pooled) & (tensor > 0)
    
    return local_max_mask

def get_top_peaks(tensor, num_peaks=2):
    local_max_mask = find_local_maxima(tensor)
    peak_values = tensor[local_max_mask]
    peak_indices = local_max_mask.nonzero(as_tuple=False)

    if peak_values.numel() < num_peaks:
        raise ValueError("Not enough peaks found.")

    sorted_values, sorted_indices = torch.sort(peak_values, descending=True)
    top_indices = peak_indices[sorted_indices[:num_peaks]]

    return top_indices

def find_local_maxima_1d(signal, kernel_size=3):
    signal_1d = signal.unsqueeze(0).unsqueeze(0)
    pad = (kernel_size // 2, kernel_size // 2)
    padded = F.pad(signal_1d, pad, mode='constant', value=float('-inf'))
    pooled = F.max_pool1d(padded, kernel_size=kernel_size, stride=1)
    is_peak = (signal_1d == pooled).squeeze()
    peak_indices = torch.nonzero(is_peak, as_tuple=False).squeeze()

    return peak_indices, signal[peak_indices]

def get_top_x_peaks_at_yz(tensor, y, z, num_peaks=2):
    signal = tensor[z, y, :]
    peak_indices, peak_values = find_local_maxima_1d(signal)

    if peak_values.numel() < num_peaks:
        raise ValueError(f"Only {peak_values.numel()} peak(s) found at (y={y}, z={z}).")

    sorted_vals, sorted_idx = torch.sort(peak_values, descending=True)
    top_x_indices = peak_indices[sorted_idx[:num_peaks]]

    return top_x_indices


if config['object'] == 'torus':
    grid_tensor = torch.abs(grid.reshaped)
    x_idx, y_idx, z_idx = grid.get_max_index()
    grid.print_slice(x_idx, y_idx, z_idx)

    top_peaks_x = get_top_x_peaks_at_yz(grid_tensor, y_idx, z_idx)
    print(f"Top x-indices at (y={y_idx}, z={z_idx}): {top_peaks_x.tolist()}")

    x_idx_1 = top_peaks_x[0]
    x_idx_2 = top_peaks_x[1]
    grid.print_slice(x_idx_1, y_idx, z_idx)
    grid.print_slice(x_idx_2, y_idx, z_idx)

if config['object'] == 'plane':
    grid_tensor = torch.abs(grid.reshaped)
    print("Grid tensor shape:", grid_tensor.shape)
    N_max = 7

    flat_tensor = grid_tensor.view(-1)
    topk_vals, topk_indices = torch.topk(flat_tensor, N_max)
    indices_3d = torch.stack(torch.unravel_index(topk_indices, grid_tensor.shape), dim=1)  # Shape: [N, 3]

    sum = 0
    for idx in indices_3d:
        z, y, x = idx.tolist()
        sum += grid.get_slice(x, y, z)[2]

    print(sum/ N_max)
